# 04 · SciSpaCy NER on Medical Conversations

Extracts biomedical named entities (diseases, chemicals/drugs) from the `utterance` column  
using **`en_ner_bc5cdr_md`** — a SciSpaCy model trained on the BC5CDR corpus, covering:
- `DISEASE` — diseases, disorders, syndromes, symptoms  
- `CHEMICAL` — drugs, chemicals, compounds  

Processing uses `nlp.pipe()` for efficient batch inference.

## 1 · Imports & configuration

In [1]:
import pandas as pd
import spacy
from tqdm.auto import tqdm
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("../data/processed")
INPUT_CSV  = DATA_DIR / "integrated_conversations.csv"
OUTPUT_CSV = DATA_DIR / "integrated_conversations_ner.csv"

# ── NER settings ─────────────────────────────────────────────────────────────
MODEL_NAME  = "en_ner_bc5cdr_md"  
BATCH_SIZE  = 256                 
N_PROCESS   = 1                   
print(f"Input  : {INPUT_CSV}")
print(f"Output : {OUTPUT_CSV}")
print(f"Model  : {MODEL_NAME}")

Input  : ..\data\processed\integrated_conversations.csv
Output : ..\data\processed\integrated_conversations_ner.csv
Model  : en_ner_bc5cdr_md


## 2 · Load dataset

In [2]:
df = pd.read_csv(INPUT_CSV)

print(f"Rows   : {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

C:\Users\nirmi\AppData\Local\Temp\ipykernel_8852\921211518.py:1: DtypeWarning: Columns (0: original_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_CSV)


Rows   : 276,172
Columns: ['dialogue_id', 'turn_id', 'speaker', 'utterance', 'source_dataset', 'original_id']


,dialogue_id,turn_id,speaker,utterance,source_dataset,original_id
0,000573958699464e9de6493b5e182fab,0,user,Prompt: I want you to be my personal mental he...,HealthChat-LMSYS,000573958699464e9de6493b5e182fab
1,000573958699464e9de6493b5e182fab,1,assistant,"NAME_2, it's nice to meet you. I'm here to hel...",HealthChat-LMSYS,000573958699464e9de6493b5e182fab
2,000573958699464e9de6493b5e182fab,2,user,I guess there's quite a few things. But I prim...,HealthChat-LMSYS,000573958699464e9de6493b5e182fab


In [3]:
# ── Sanity-check the utterance column ────────────────────────────────────────
missing = df["utterance"].isna().sum()
print(f"Missing utterances : {missing}")

# Fill NaN with empty string so nlp.pipe() never receives None
df["utterance"] = df["utterance"].fillna("").astype(str)
print(f"Sample utterance   : {df['utterance'].iloc[2][:120]}")

Missing utterances : 198
Sample utterance   : I guess there's quite a few things. But I primarily struggle with cleanliness 


## 3 · Load SciSpaCy model

In [7]:
nlp = spacy.load(MODEL_NAME)

for pipe in ["parser", "lemmatizer"]:
    if pipe in nlp.pipe_names:
        nlp.disable_pipe(pipe)

print(f"Loaded  : {MODEL_NAME}")
print(f"Pipeline: {nlp.pipe_names}")
print(f"Entities: {nlp.get_pipe('ner').labels}")

OSError: [E050] Can't find model 'en_ner_bc5cdr_md'. It doesn't seem to be a Python package or a valid path to a data directory.

## 4 · Batch NER extraction

`nlp.pipe()` streams utterances in batches — far faster than calling `nlp(text)` row-by-row.

In [ ]:
def extract_entities(docs):
    """
    Given an iterable of spaCy Doc objects, return two parallel lists:
      - entity_texts  : list of entity surface strings per doc
      - entity_labels : list of entity label strings per doc
    """
    entity_texts  = []
    entity_labels = []

    for doc in docs:
        texts  = [ent.text  for ent in doc.ents]
        labels = [ent.label_ for ent in doc.ents]
        entity_texts.append(texts)
        entity_labels.append(labels)

    return entity_texts, entity_labels


# ── Run nlp.pipe() over all utterances ───────────────────────────────────────
utterances = df["utterance"].tolist()

print(f"Processing {len(utterances):,} utterances  "
      f"(batch_size={BATCH_SIZE}) …")

# Wrap the generator in tqdm for a progress bar
docs = nlp.pipe(
    utterances,
    batch_size=BATCH_SIZE,
    n_process=N_PROCESS
)

docs_with_progress = tqdm(docs, total=len(utterances), desc="NER")

entity_texts, entity_labels = extract_entities(docs_with_progress)

print("Done.")

## 5 · Attach results to the dataframe

In [ ]:
# Store as lists (native Python objects in each cell)
df["extracted_entities"] = entity_texts
df["entity_labels"]      = entity_labels

# Quick stats
has_entity = df["extracted_entities"].apply(lambda x: len(x) > 0)
total_ents = df["extracted_entities"].apply(len).sum()

print(f"Utterances with ≥1 entity : {has_entity.sum():,} "
      f"({has_entity.mean()*100:.1f}%)")
print(f"Total entities extracted  : {total_ents:,}")

df[["utterance", "extracted_entities", "entity_labels"]].head(10)

## 6 · Entity frequency breakdown

In [ ]:
from collections import Counter

# Flatten all (entity, label) pairs for counting
all_pairs = [
    (ent, lbl)
    for ents, lbls in zip(entity_texts, entity_labels)
    for ent, lbl in zip(ents, lbls)
]

# Label distribution
label_counts = Counter(lbl for _, lbl in all_pairs)
print("Entity label distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:<12} {count:>7,}")

print()

# Top 20 most frequent entities
top_entities = Counter(ent.lower() for ent, _ in all_pairs).most_common(20)
print("Top 20 entities:")
for ent, cnt in top_entities:
    print(f"  {cnt:>6,}  {ent}")

## 7 · Save output CSV

In [ ]:
# Convert list columns to pipe-delimited strings for clean CSV storage
df_out = df.copy()
df_out["extracted_entities"] = df_out["extracted_entities"].apply(
    lambda x: " | ".join(x) if x else ""
)
df_out["entity_labels"] = df_out["entity_labels"].apply(
    lambda x: " | ".join(x) if x else ""
)

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"✅  Saved {len(df_out):,} rows → {OUTPUT_CSV}")
df_out[["utterance", "extracted_entities", "entity_labels"]].head(5)

---
### Summary

| Step | Detail |
|------|--------|
| Model | `en_ner_bc5cdr_md` (SciSpaCy, BC5CDR corpus) |
| Entity types | `DISEASE`, `CHEMICAL` |
| Processing | `nlp.pipe()` with batch_size=256 |
| Output columns | `extracted_entities` (pipe-delimited text), `entity_labels` (pipe-delimited labels) |
| Output file | `data/processed/integrated_conversations_ner.csv` |